In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
SEC filings downloader & converter (stable version)
- Auto CIK zero-padding (10 digits)
- Resume support: skip already downloaded files
- Robust logging + summary CSV
- HTML 清理 (移除 mso-style, text-autospace, SEC 外部资源)
- Output to ~/Downloads/SEC_EDGAR_Fillings/processed_pdf
"""

import os
import sys
import re
import json
import time
import csv
import logging
import requests
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Semaphore

# Optional deps
try:
    from weasyprint import HTML

    WEASYPRINT_AVAILABLE = True
except Exception:
    WEASYPRINT_AVAILABLE = False

try:
    import markdownify

    MARKDOWNIFY_AVAILABLE = True
except Exception:
    MARKDOWNIFY_AVAILABLE = False

# ================== 配置 ==================
DOWNLOAD_DIR = os.path.expanduser("~/Downloads/SEC_EDGAR_Fillings/processed_pdf")
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# 加载 .env 环境变量
load_dotenv()
USER_EMAIL = os.getenv("USER_EMAIL")
if not USER_EMAIL:
    raise ValueError("❌ 请在 .env 文件中设置 USER_EMAIL")

HEADERS = {"User-Agent": USER_EMAIL}

MAX_WORKERS = min(8, (os.cpu_count() or 4))
MAX_CONCURRENT_REQUESTS = 3
REQUEST_TIMEOUT = 30
RETRY_COUNT = 3
RETRY_BACKOFF = 2.0

LOG_FILE = os.path.join(DOWNLOAD_DIR, "sec_edgar.log")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler(sys.stdout),
    ],
)
logger = logging.getLogger("sec_edgar")

# 抑制 weasyprint 的 warning 日志
logging.getLogger("weasyprint").setLevel(logging.ERROR)

SUMMARY_CSV = os.path.join(DOWNLOAD_DIR, "processing_summary.csv")

# ================== 工具函数 ==================
sema = Semaphore(MAX_CONCURRENT_REQUESTS)


def sanitize_filename(s: str, max_len: int = 160) -> str:
    if not s:
        return ""
    safe = s.replace("/", "_").replace("\\", "_").replace(":", "_")
    safe = "".join(ch for ch in safe if ch.isprintable())
    return safe[:max_len]


def make_accession_clean(accession: str) -> str:
    return accession.replace("-", "")


def ensure_cik_padded(cik: str) -> str:
    """确保 CIK 为 10 位"""
    return str(int(cik)).zfill(10)


def http_get_with_retries(
    url, headers, timeout=REQUEST_TIMEOUT, max_retries=RETRY_COUNT
):
    attempt = 0
    while True:
        try:
            with sema:
                resp = requests.get(url, headers=headers, timeout=timeout)
            resp.raise_for_status()
            return resp.content
        except Exception as e:
            attempt += 1
            if attempt > max_retries:
                raise
            backoff = RETRY_BACKOFF ** (attempt - 1)
            logger.warning(
                f"请求失败 {url}: {e}, {backoff}s 后重试 ({attempt}/{max_retries})"
            )
            time.sleep(backoff)


def safe_write(path: str, content: bytes, mode="wb"):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, mode) as f:
        f.write(content)


def clean_html(raw_html: str) -> str:
    """清理 SEC HTML 中的垃圾样式和外链"""
    raw_html = re.sub(r'mso-[^:]+:[^;"]+;?', "", raw_html, flags=re.IGNORECASE)
    raw_html = re.sub(r'text-autospace:[^;"]+;?', "", raw_html, flags=re.IGNORECASE)
    raw_html = re.sub(r"color:\s*windowtext;?", "", raw_html, flags=re.IGNORECASE)
    raw_html = re.sub(
        r'border[^:]+:\s*[^;]*windowtext[^;"]*;?', "", raw_html, flags=re.IGNORECASE
    )
    raw_html = re.sub(r"<link[^>]+stylesheet[^>]+>", "", raw_html, flags=re.IGNORECASE)
    raw_html = re.sub(
        r'<img[^>]+src="https://www.sec.gov[^>]+>', "", raw_html, flags=re.IGNORECASE
    )
    return raw_html


def convert_html_to_pdf(html_bytes: bytes, base_url: str) -> bytes:
    if not WEASYPRINT_AVAILABLE:
        raise RuntimeError("WeasyPrint 不可用")
    html_text = html_bytes.decode("utf-8", errors="ignore")
    html_text = clean_html(html_text)
    return HTML(string=html_text, base_url=base_url).write_pdf()


def txt_to_pdf(txt: str) -> bytes:
    if not WEASYPRINT_AVAILABLE:
        raise RuntimeError("WeasyPrint 不可用")
    html = f"<html><body><pre>{txt}</pre></body></html>"
    return HTML(string=html).write_pdf()


# ================== 主处理 ==================
def process_filing(cik_padded, row, total, index):
    form, primary_doc, date, accession = row
    accession_clean = make_accession_clean(accession)
    form_folder = sanitize_filename(form.replace(" ", "_"))
    output_folder = os.path.join(DOWNLOAD_DIR, form_folder)
    os.makedirs(output_folder, exist_ok=True)

    base_name = f"{form}_{date}_{accession_clean}"
    ext = os.path.splitext(primary_doc)[1].lower()
    out_path = os.path.join(output_folder, f"{sanitize_filename(base_name)}.pdf")

    if os.path.exists(out_path):
        logger.info(f"[{index}/{total}] 已存在，跳过: {primary_doc}")
        return ("skipped", out_path)

    try:
        url = f"https://www.sec.gov/Archives/edgar/data/{int(cik_padded)}/{accession_clean}/{primary_doc}"
        logger.info(f"[{index}/{total}] 下载: {url}")
        content = http_get_with_retries(url, HEADERS)

        if ext == ".pdf":
            safe_write(out_path, content)
            status = "pdf_saved"
        elif ext in (".htm", ".html", ".xml"):
            try:
                pdf_bytes = convert_html_to_pdf(
                    content, base_url=os.path.dirname(url) + "/"
                )
                safe_write(out_path, pdf_bytes)
                status = "html_to_pdf"
            except Exception as e:
                logger.warning(f"HTML 转 PDF 失败，保存原始 HTML: {e}")
                html_path = out_path.replace(".pdf", ".html")
                safe_write(html_path, content)
                status = "html_saved"
        elif ext == ".txt":
            try:
                text = content.decode("utf-8", errors="ignore")
                pdf_bytes = txt_to_pdf(text)
                safe_write(out_path, pdf_bytes)
                status = "txt_to_pdf"
            except Exception:
                txt_path = out_path.replace(".pdf", ".txt")
                safe_write(txt_path, content, mode="wb")
                status = "txt_saved"
        else:
            other_path = out_path.replace(".pdf", ext)
            safe_write(other_path, content)
            status = "unknown_saved"

        logger.info(f"[{index}/{total}] 完成: {primary_doc} -> {status}")
        return (status, out_path)

    except Exception as e:
        logger.error(f"[{index}/{total}] 错误: {primary_doc}: {e}")
        return ("error", None)


# ================== 主函数 ==================
def main():
    CIK = "1708176"  # 修改为目标公司 CIK
    cik_padded = ensure_cik_padded(CIK)

    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
    logger.info(f"获取文件列表: {url}")
    content = http_get_with_retries(url, HEADERS)
    data = json.loads(content.decode("utf-8", errors="ignore"))

    filings = data.get("filings", {}).get("recent", {})
    rows = list(
        zip(
            filings.get("form", []),
            filings.get("primaryDocument", []),
            filings.get("filingDate", []),
            filings.get("accessionNumber", []),
        )
    )

    total = len(rows)
    logger.info(f"共 {total} 个文件")

    results = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(process_filing, cik_padded, row, total, i + 1): row
            for i, row in enumerate(rows)
        }
        for fut in as_completed(futures):
            row = futures[fut]
            try:
                status, path = fut.result()
                results.append([*row, status, path])
            except Exception as e:
                logger.error(f"处理失败 {row}: {e}")
                results.append([*row, "error", None])

    with open(SUMMARY_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["Form", "File", "Date", "Accession", "Status", "Path"])
        writer.writerows(results)
    logger.info(f"进度已保存到 {SUMMARY_CSV}")


if __name__ == "__main__":
    main()

2025-10-17 23:39:19,134 [INFO] 获取文件列表: https://data.sec.gov/submissions/CIK0001708176.json
2025-10-17 23:39:19,378 [INFO] 共 679 个文件
2025-10-17 23:39:19,379 [INFO] [2/679] 已存在，跳过: ea025978602-sc13e3a5_hall.htm
2025-10-17 23:39:19,379 [INFO] [1/679] 已存在，跳过: xslSCHEDULE_13D_X01/primary_doc.xml
2025-10-17 23:39:19,381 [INFO] [4/679] 已存在，跳过: ef20056106_8k.htm
2025-10-17 23:39:19,381 [INFO] [3/679] 已存在，跳过: ef20056539_8k.htm
2025-10-17 23:39:19,381 [INFO] [5/679] 已存在，跳过: ea0257820-sc13e3a4_hall.htm
2025-10-17 23:39:19,382 [INFO] [6/679] 已存在，跳过: xslSCHEDULE_13D_X01/primary_doc.xml
2025-10-17 23:39:19,382 [INFO] [7/679] 已存在，跳过: ef20055719_defa14a.htm
2025-10-17 23:39:19,387 [INFO] [8/679] 已存在，跳过: ef20055719_8k.htm
2025-10-17 23:39:19,387 [INFO] [9/679] 已存在，跳过: ef20055661_defa14a.htm
2025-10-17 23:39:19,387 [INFO] [10/679] 已存在，跳过: ef20055661_8k.htm
2025-10-17 23:39:19,388 [INFO] [11/679] 已存在，跳过: xslSCHEDULE_13D_X01/primary_doc.xml
2025-10-17 23:39:19,389 [INFO] [12/679] 已存在，跳过: ef20055327_8k.htm